# WBG MetricsInMotion - Building Digital Twin Models for the Energy Transition

## What is the right level of investment on renewable infrastructure to match a given demand?

## The Scenario

Meet a typical Luxembourg household: two working parents, two school-age children, a gas boiler
for heating, no electric vehicle. Their electricity bill has climbed sharply since 2022.
They are wondering whether investing in rooftop solar panels — and possibly a battery —
would save money over the next decade.

This notebook will work through that question rigorously, using **real consumption data from
their smart meter**, **historical Luxembourg solar records**, and **actual installation costs
and tariffs**. At the end, you will have a data-driven answer — and an understanding of the
assumptions that drive it.

---

## What This Notebook Covers

The analysis follows five steps in sequence:

| Section | What We Do |
|---|---|
| **1. Understanding Consumption** | Examine when and how much electricity this household uses. This defines what we are trying to offset. |
| **2. Solar Potential** | Explore how much solar energy this specific roof can generate, and how it varies year to year. |
| **3. Installation Configurations** | Simulate every combination of panel count and battery size simultaneously, building a complete map of outcomes. |
| **4. Annual Electricity Bill** | Calculate the current bill and how it changes under each configuration, accounting for feed-in revenue. |
| **5. Price Forecast & ROI** | Project prices over 10 years, input installation costs, and identify which configurations are financially viable. |

---

## What You Need to Do

This notebook is designed to be run **from top to bottom**, one cell at a time.
- Identify **Code Cells**: Look for cells with a bracket [ ] in the margin on the left. Each code cell has a Play Button in the top left corner.  
  You do not need to read or understand the code (though of course you can take a look!).   
  Some code cells are 'collapsed' and all you see is the text "show code". You need to run these cells too.  
- **To run the Cell: Click the Play Button**.   
Or you can select the cell and  press Shift + Enter on your keyboard (works on Windows and Mac) to execute this cell AND move to the next.
- Monitor Progress: While the code is calculating, the icon border will spin.
  - Success: A green checkmark and a number (e.g., [1]) will appear when the task is finished.
  - Error: If something goes wrong, a red exclamation mark or an error message will appear at the bottom of the cell. The most likely reason for an error is that you skipped previous cells! Please check you executed every cell in order.  

- **A few cells are intended for you to edit.** These are clearly marked:
  <div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
  <b style="color: #2eb886;">✎ Action Required:</b> Modify the parameters below to test the network sensitivity.
  </div> You will be able to:  
  - Choose a year of solar data to use (good/medium/bad)  
  - Set the number of panels to explore  
  - Forecast how electricity prices might change  
  - Input installation cost estimates  

- **Interactive charts** appear after certain cells. Use the dropdowns, sliders, and checkboxes
  to explore the results — there is nothing to break.

> **A note on the numbers:** All financial figures use Luxembourg tariff rates and subsidy rules. The 10-year forecast is based on *your assumptions* — the notebook will allow to investigate how sensitive the result is to those choices.

### Setup
Run the notebook one cell at a time.  
You do not need to modify anything at this stage.

In [3]:
! git clone https://github.com/richardconnors/pv_battery_simple_demo.git
! gcd /content/pv_battery_simple_demo
! gpip install -f requirements.txt

SyntaxError: invalid syntax (3650636996.py, line 3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Patch
import itertools
from ipywidgets import interact, Dropdown, FloatSlider, Output, VBox, HTML, IntSlider
import pandas as pd
import numpy as np
from IPython.display import display


# --- Internal Library Imports ---
from pod_tools.loaders import load_smart_meter_export
from pod_tools.extras import get_public_holidays, load_school_holidays, display_monthly_summary_table,calculate_annual_bill
from pod_tools.pv import get_cached_statistical_year, load_pv_data
from pod_tools.simulation import simulate_battery_greedy, apply_heat_pump_model

from pod_tools.plotting import (
    plot_total_daily_scatter,
    plot_all_data_scatter,
    interactive_weekly_PVbattery,
    interactive_consumption_histograms,
    plot_daily_pv_percentiles,
    plot_marginal_gains_dynamic,
    plot_absolute_position_heatmap,
    plot_parameter_space_heatmaps,
    plot_investment_metrics_heatmaps,
    plot_trend_with_cloud_scatter,
    plot_monthly_breakdown,
    plot_roi_heatmap)
# Dynamic Pathing
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "pod_data"
OTHER_DATA_DIR = BASE_DIR / "other_data"
PROCESSED_DIR = BASE_DIR / "processed_data"
PV_DIR = BASE_DIR / "pv_data"

POD_ID = "WB_POD_WorkingFamily" # working family house consumption
FAMILY_POD_FILE = OTHER_DATA_DIR / "WB_POD_WorkingFamily.csv"
# Load the empirical data
df_15m = load_smart_meter_export(FAMILY_POD_FILE)
asset_name = "Working Family Home"
NUM_PANELS = 1; # for all baseline calculation

### Parameters
`TARGET_YEAR` sets which year of consumption data drives the analysis.

`PANEL_KWP` is the power rating of the panels we will model (0.46 'kilowatts peak' is the current standard).

`BATTERY_KWH` is the battery size in kilowatt hours used in the initial simulations below (other battery sizes will be tested later).

In [ ]:
# Put in some parameter choices for the first run through.

TARGET_YEAR = 2024; # choose 2023 or 2024
PANEL_KWP = 0.46;
BATTERY_KWH = 10

In [ ]:
# 1. Isolate the target year without destroying the master DataFrame
df_15m_target = df_15m[df_15m.index.year == TARGET_YEAR].copy()
# --- Establish the hourly consumption index ---
cons_h = df_15m_target["energy_kwh"].resample("1h").sum()

# 2. Resample to daily totals
df_daily = df_15m_target["energy_kwh"].resample("D").sum().reset_index()
df_daily.rename(columns={"timestamp": "date", "energy_kwh": "total_consumption"}, inplace=True)
df_daily["date"] = df_daily["date"].dt.normalize()

# 3. Load & Merge Public Holidays
# (Using target_years ensures we only fetch what we need)
target_years = df_daily['date'].dt.year.unique().tolist()
pub_holidays = get_public_holidays(target_years)
df_daily = df_daily.merge(pub_holidays, on="date", how="left")

# 4. Load & Merge School Holidays
sch_file = OTHER_DATA_DIR / "SchoolHolidays.csv"
if sch_file.exists():
    sch_holidays = load_school_holidays(sch_file)
    df_daily = df_daily.merge(sch_holidays, on="date", how="left")
else:
    print(f" '{sch_file.name}' not found. Skipping school holiday tagging.")
    df_daily["is_school_holiday"] = False

# 5. Clean boolean flags
df_daily["is_public_holiday"] = df_daily["is_public_holiday"] == True
df_daily["is_school_holiday"] = df_daily["is_school_holiday"] == True

# 6. Classify Day Types
def classify_day(row):
    if row["is_public_holiday"]: return "Public Holiday"
    if row["is_school_holiday"]: return "School Holiday"
    if row["date"].dayofweek == 5: return "Saturday"
    if row["date"].dayofweek == 6: return "Sunday"
    return "Weekday"

df_daily["day_type"] = df_daily.apply(classify_day, axis=1)

# Precompute sunrise/sunset times for TARGET_YEAR — cached to parquet
from pod_tools.extras import get_daylight_hours
df_daylight = get_daylight_hours(TARGET_YEAR, OTHER_DATA_DIR / "cache")

In [ ]:
# PV DATA LOADING
# All scenarios are for 1 panel (PANEL_KWP). Scaling by number
# of panels happens at the point of simulation, not here.
# =============================================================

# --- Load full historical archive (for exploration plots only) ---
# DataFrame, column: total_pv_kwh. Spans all available years (~2005-2023).
# Do NOT use this directly in simulation — it contains multiple years
# and the timestamps do not align with TARGET_YEAR.
pv_history, _ = load_pv_data(
    pod_number=POD_ID,
    panels_per_array=1,
    panel_kwp=PANEL_KWP,
    base_dir=PV_DIR,
    verbose=False
)
# --- Build synthetic statistical scenarios aligned to TARGET_YEAR ---
# These are constructed day-by-day from historical percentiles.
# Each day is independently drawn — there is no multi-day autocorrelation.
# P25 scenario is MORE pessimistic than any real bad year ever was.
_pv_synth_mean_df, _ = get_cached_statistical_year(POD_ID, PV_DIR, 1, PANEL_KWP, TARGET_YEAR, "mean")
_pv_synth_good_df, _ = get_cached_statistical_year(POD_ID, PV_DIR, 1, PANEL_KWP, TARGET_YEAR, "good")
_pv_synth_bad_df,  _ = get_cached_statistical_year(POD_ID, PV_DIR, 1, PANEL_KWP, TARGET_YEAR, "bad")
# Extract as Series, aligned to cons_h index
pv_synth_mean = _pv_synth_mean_df["total_pv_kwh"].reindex(cons_h.index).fillna(0)
pv_synth_good = _pv_synth_good_df["total_pv_kwh"].reindex(cons_h.index).fillna(0)
pv_synth_bad  = _pv_synth_bad_df["total_pv_kwh"].reindex(cons_h.index).fillna(0)

# --- Extract specific historical years aligned to TARGET_YEAR ---
# These use real weather data with authentic seasonal autocorrelation.
# Leap year handling: Feb 29 source data is dropped if target is non-leap;
# Feb 29 of a leap target year is interpolated from Feb 28 / Mar 1.
def _extract_hist_year(history_df, source_year, target_index):
    """
    Extracts one calendar year from pv_history and aligns its
    month/day/hour structure onto target_index, handling leap years correctly.
    Always returns a Series.
    """
    target_year = target_index.year[0]
    src = history_df[history_df.index.year == source_year].copy()

    # Remap timestamps: keep month/day/hour, swap year to target
    def _remap(ts):
        try:
            return ts.replace(year=target_year)
        except ValueError:
            # Feb 29 in source but target is non-leap — mark for removal
            return pd.NaT

    src.index = src.index.map(_remap)
    src = src[src.index.notna()]  # drop the Feb 29 if target is non-leap

    # Squeeze to Series
    s = src.iloc[:, 0] if isinstance(src, pd.DataFrame) else src

    # Reindex to full target hourly grid; interpolate any gaps (e.g. leap target,
    # non-leap source means Feb 29 is missing — interpolate from neighbours)
    s = s.reindex(target_index)
    s = s.interpolate(method="time").fillna(0)
    return s

# Identify which years are bad/median/good from the history
_annual_pv = pv_history.resample("YE").sum().iloc[:, 0]
_annual_pv.index = _annual_pv.index.year
_hist_bad_year  = int(_annual_pv.idxmin())
_hist_good_year = int(_annual_pv.idxmax())
_hist_med_year  = int((_annual_pv - _annual_pv.median()).abs().idxmin())

pv_hist_bad  = _extract_hist_year(pv_history, _hist_bad_year,  cons_h.index)
pv_hist_med  = _extract_hist_year(pv_history, _hist_med_year,  cons_h.index)
pv_hist_good = _extract_hist_year(pv_history, _hist_good_year, cons_h.index)

# --- Weather Loading  ---
# Load weather (ensure index is aligned with cons_h)
weather_file = OTHER_DATA_DIR / f"real_weather_{TARGET_YEAR}.csv"
df_weather = pd.read_csv(weather_file, parse_dates=['date'], index_col='date')
df_weather.index = df_weather.index.tz_localize(None) # Match your consumption TZ

# Align temperature to your consumption hourly index
temp_h = df_weather['temp_air'].reindex(cons_h.index).ffill()


## 1. Understanding Current Consumption

In the scatter plot below, each dot is one day's total electricity use (checkbox allows day type coloring).

In [ ]:
plot_total_daily_scatter(df_daily, day_type_col="day_type")

Look to see if there is systematic seasonal variation (higher winter consumption?), lower consumption during summer holidays, any weekday/weekend split, and any outliers.

We can also look for consumption patterns within each day; different days may have different consumption schedules.  
The heatmap below shows **median consumption by hour and day of week** across the full year.

In [ ]:
df_15m['hour']        = df_15m.index.hour
df_15m['day_of_week'] = df_15m.index.dayofweek
heatmap_data = df_15m.pivot_table(
    index='hour',
    columns='day_of_week',
    values='energy_kwh',
    aggfunc='median'
)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
heatmap_data.columns = [day_names[i] for i in heatmap_data.columns]

plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data.T, cmap='turbo', cbar_kws={'label': 'Median kWh'})
plt.title(f"Median Consumption Profile ({TARGET_YEAR})\n{asset_name}")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()

Red/orange cells indicate when this household consistently draws the most power.   
You can see a minor peak each weekday morning 7am-8am. However, most consumption occurs in evening peaks.  
Note that PV generation peaks at midday and will not coincide with peak demand which is often after sunset.

**Consumption after sunset**  
Toggle the checkboxes to compare typical total daily consumption against nocturnal consumption (sunset to sunrise).  

In [ ]:
interactive_consumption_histograms(cons_h, TARGET_YEAR, df_daylight)

Many days have low overnight consumption (below 3kWh). However, shorter daylight hours in Winter results in higher 'nocturnal' consumption. The nocturnal portion is load that PV *cannot* cover directly — it must either come from the grid or from a battery charged during the day. A large nocturnal fraction is a primary justification for battery storage.

## 2. Solar Potential

Not all roofs are equal.   
In Luxembourg the **Solar Cadaster** maps the annual irradiation potential of every roof surface.

You can visit the [solar cadaster online](https://map.geoportail.lu/theme/main?version=3&X=668111&Y=6444534&zoom=18&rotation=0&features=&lang=fr&layers=1813&opacities=1&time=&bgLayer=topo_bw_jpeg&serial=) here


In the left plot - dark red indicates high suitability, yellow indicates shading or poor orientation. For this house, only the South-West-facing section is deemed viable. This roof section has   
Azimuth (compass bearing)  = 224 degrees  
Slope = 39 degrees (from horizontal)  
The right image shows how many standard PV panels can be tesselated on the usable roof.  
These parameters are fixed inputs to the PV simulation.

<div style="display: flex; justify-content: center; align-items: stretch; height: 350px;">
    <img src="https://github.com/richardconnors/pv_battery_simple_demo/blob/main/images/SolarCadasterandPanels.png?raw=1" style="width: 100%; height: 100%; margin-right: 5px; object-fit: contain;">
</div>



### Simulating PV Generation Including Natural Variability

Our PV simulation data comes from PVGIS — the EU's photovoltaic geographic information system — using Luxembourg-specific historical weather records from 2005 to 2023. Results below are for **one panel (0.46 kWp)** on this specific roof (using latitude, longitude, azimuth and slope).

[PVGIS Simulator Online](https://re.jrc.ec.europa.eu/pvg_tools/en/tools.html)

Year-to-year variability is significant: a bad year can produce 20% less than a good year. The plots below characterise this range from the historical record

In [ ]:
monthly_pv = pv_history.resample('ME').sum()
plt.figure(figsize=(8, 5))
for year, label, color in zip(
    [_hist_bad_year,   _hist_med_year,    _hist_good_year],
    ['Bad Year (Min)', 'Median Year',     'Good Year (Max)'],
    ['#d62728',        '#7f7f7f',         '#2ca02c']
):
    year_data = monthly_pv[monthly_pv.index.year == year]
    plt.plot(range(1, 13), year_data.values, marker='o',
             label=f"{year}: {label}", color=color, linewidth=2)

plt.title("Good/Bad PV Production Comparison (Monthly Totals)", fontsize=14)
plt.xlabel("Month", fontsize=12)
plt.ylabel("PV Production (kWh)", fontsize=12)
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                           'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
summary_df = pd.DataFrame({
    "Year": [_hist_bad_year, _hist_med_year, _hist_good_year],
    "kWh/panel/yr": [
        f"{pv_hist_bad.sum():.1f}",
        f"{pv_hist_med.sum():.1f}",
        f"{pv_hist_good.sum():.1f}",
    ],
    "kWh/kWp/yr": [
        f"{pv_hist_bad.sum()  / PANEL_KWP:.1f}",
        f"{pv_hist_med.sum()  / PANEL_KWP:.1f}",
        f"{pv_hist_good.sum() / PANEL_KWP:.1f}",
    ],
}, index=["Bad", "Median", "Good"])
summary_df.index.name = "Scenario"

ROW_COLORS = {
    "Bad":    "#d62728",
    "Median": "#555555",
    "Good":   "#2ca02c",
}

def _style_scenario(row):
    color = ROW_COLORS.get(row.name, "#111")
    return [f"color: {color}; font-weight: bold"] * len(row)

display(
    summary_df.style
    .apply(_style_scenario, axis=1)
    .set_caption("Historical PV Production by Scenario (1 panel)")
    .set_table_styles([
        {"selector": "table",
         "props": [("background-color", "white"), ("border-collapse", "collapse")]},
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"), ("color", "#111"),
                   ("text-align", "left"), ("padding-bottom", "6px"),
                   ("background-color", "white")]},
        {"selector": "th",
         "props": [("text-align", "center"), ("border-bottom", "2px solid #111"),
                   ("padding", "6px 16px"), ("color", "#111"),
                   ("background-color", "white")]},
        {"selector": "td",
         "props": [("text-align", "center"), ("padding", "5px 16px"),
                   ("background-color", "white")]},
    ])
)

The worst year on record is 2013 where 1 panel would have produced only 418.7kWh/year.
However note that August 2013 was better than 2006!  
PV energy production is highly variable day-to-day. Combine this with household-specific consumption patterns, holiday timing and other characteristics, it can be useful to do individual level analysis.

### PV Scenario Selection

**Choose one** year of PV data by editing `PV_SCENARIO_ACTIVE` in the cell below  
This affects all the following simulations in the notebook.

<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Specify which PV data to use in the analysis (bad, med or good year).
</div>

In [ ]:
PV_SCENARIO_ACTIVE = pv_hist_med

# =============================================================
# you can choose any of the following scenarios
#   pv_hist_bad    Worst full year on record
#   pv_hist_med    Median full year on record
#   pv_hist_good   Best full year on record
# =============================================================

# This is just a text label that gets used in some plot titles and tables
# it is useful to update it to match the scenario you specified above.
PV_SCENARIO_LABEL  = "Historical Good (Best Year)"

### How Many Panels?

<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Set `NUM_PANELS` in the cell below and run it. Full roof coverage is 52 panels.
</div>

In [ ]:
NUM_PANELS = 15

The scatter plot below show daily consumption (in grey) and the total daily PV production using your chosen settings for   
`PV_SCENARIO_ACTIVE`  
`NUM_PANELS`

Keep in mind even when the daily PV output is high it is unlikely to coincide with the hours of highest demand.

In [ ]:
# Uses the active scenario set above. Change PV_SCENARIO_ACTIVE to explore others.
pv_to_plot = (NUM_PANELS * PV_SCENARIO_ACTIVE).to_frame(name="total_pv_kwh")
plot_trend_with_cloud_scatter(df_15m_target, pv_to_plot, pod_name=asset_name)

The above plot will change depening on the PV scenario selected and on the number of panels.  
It shows the dynamic variability of consumption and generation throughout the year.

Recall that matching *totals* is not the same as matching *timing* — that mismatch is explored in the next plot that shows the **average hourly generation and consumption profiles** in January and June.

In [ ]:

# --- 1. Prepare the aggregated Data ---
# Combine aligned hourly series into a single DataFrame
df_combined = pd.DataFrame({
    "consumption_kWh": cons_h,
    "pv_kWh": NUM_PANELS*PV_SCENARIO_ACTIVE
})
df_combined = df_combined[df_combined.index.dayofweek < 5]
# Extract temporal dimensions
df_combined["month"] = df_combined.index.month
df_combined["hour"] = df_combined.index.hour

# Calculate average profile per hour, per month
monthly_avg = df_combined.groupby(["month", "hour"]).mean().reset_index()

# --- 2. Plotting Logic ---
# --- Specify Months to Plot ---
months_to_plot = [1, 6] # January and June

# --- Adjust Subplot Layout (1 row, 2 columns) ---
fig, axes = plt.subplots(1, len(months_to_plot), figsize=(12, 5), sharey=True)

# If only one month, axes might not be an array, ensure it is
if len(months_to_plot) == 1:
    axes = [axes]

# --- Loop Through Selected Months ---
for i, m in enumerate(months_to_plot):
    # Select data for the current month
    dfm = monthly_avg[monthly_avg["month"] == m]

    # Check if data exists for this month
    if dfm.empty:
        print(f"No data found for month {m}. Skipping plot.")
        continue

    ax = axes[i] # Select the correct subplot

    # --- Plot Consumption as Filled Area ---
    ax.fill_between(
        dfm["hour"],              # x-values
        dfm["consumption_kWh"],   # y-values
        step="post",              # Match the step plot style
        label="Consumption",
        color="grey",             # Fill color
        alpha=0.3                 # Reduced transparency to let grid show
    )

    # Add an edge for clarity
    ax.step(dfm["hour"], dfm["consumption_kWh"], where="post", color="grey", lw=1.5)

    # --- Plot PV as Line ---
    ax.step(dfm["hour"], dfm["pv_kWh"], where="post", label="PV Generation", color="tab:green", lw=3)

    # --- Plot Formatting ---
    ax.set_title(pd.to_datetime(str(m), format="%m").strftime("%B"), fontsize=14)
    ax.set_xlim(0, 23)
    ax.set_xticks(range(0, 24, 3)) # Ticks every 3 hours
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Hour of Day")

    # Only set y-label on the first plot
    if i == 0:
        ax.set_ylabel("Average Energy (kWh/hour)")

# --- Add Legend ---
handles, labels = axes[0].get_legend_handles_labels()
# Add the fill handle (need to create a dummy patch)
handles.insert(0, Patch(facecolor='grey', alpha=0.3, edgecolor='grey'))
labels.insert(0, 'Consumption')

# Remove duplicate labels if needed (e.g., from ax.step for consumption edge)
unique_labels = {}
final_handles = []
final_labels = []
for handle, label in zip(handles, labels):
    if label not in unique_labels:
        unique_labels[label] = handle
        final_handles.append(handle)
        final_labels.append(label)

fig.legend(final_handles, final_labels, loc='upper right', bbox_to_anchor=(0.95, 0.95))
fig.suptitle('Average Daily Profile: Consumption vs. PV Generation', fontsize=16)
plt.tight_layout(rect=[0, 0, 0.95, 0.95]) # Adjust layout for legend/title
plt.show()

Notice how the PV generation peak (midday) occurs after the small morning consumption peak, and before the main evening consumption period. This temporal mismatch is the fundamental challenge solar-only installations face, and the reason battery storage adds value in a renewable installation.

### Weekly Energy Flow

The interactive plot below shows hourly energy flows for a single week.   
<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Use the "Season" dropdown to compare winter, spring, and summer behaviour. Toggle the checkboxes in order to build up the full picture layer by layer:
</div>

1. PV production  — how much solar is generated and available for use/storage
2. PV usage — the portion consumed directly (gren) vs stored/exported (pink)
3. Battery usage — discharge provides for consumption after sunset

Changing the **Season** dropdown illustrates the impact of shorter day length, worse weather and lower solar irradiation.

This plot uses your previous settings for `PV_SCENARIO_ACTIVE`, `NUM_PANELS`, and `BATTERY_KWH`.

In [ ]:
results_greedy = simulate_battery_greedy(
    cons_h,
    NUM_PANELS*PV_SCENARIO_ACTIVE,
    battery_capacity_kWh=BATTERY_KWH,
    charge_eff=0.9
)
interactive_weekly_PVbattery(results_df=results_greedy)


Notice that in Winter the low PV production is not sufficient to charge the battery and cover the nighttime consumption.  
In the Summer (if the weather is good) PV production can cover all daytime consumption plus fully charge the battery, and the shorter nightime hours allow the house to be totally self-sufficient.


## 3. Exploring the trade-offs of different installation configurations

Rather than evaluating one configuration at a time, in the cell below we simulate **many combinations** of panel count and battery size simultaneously. This takes a couple of seconds.  
The results are used by all the heatmaps below.


> **To explore a different PV scenario:** go back to the scenario selection cell, change `PV_SCENARIO_ACTIVE`, then re-run that cell and all cells below it.

In [ ]:
# Calculate all conbinations of panel and battery parameters
battery_domain = [0, 5, 10, 15, 20]
panel_domain = np.arange(0, 56, 4)

# Dictionary to store the evaluated solutions mapping (panels, battery) -> DataFrame
results_grid = {}

for p, b in itertools.product(panel_domain, battery_domain):
    results_grid[(p, b)] = simulate_battery_greedy(
        cons_h,
        p * PV_SCENARIO_ACTIVE,
        battery_capacity_kWh=b,
        charge_eff=0.9
    )

### Monthly Source Breakdown

The stacked bars show what fraction of each month's consumption came from direct PV, battery discharge, or the grid. Summer months should show high PV self-consumption; winter months reveal the irreducible grid dependence.  

<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Use the "Season" dropdown to compare winter, spring, and summer behaviour. Toggle the checkboxes in order to build up the full picture layer by layer:
</div>

Use the dropdowns to compare different panel/battery configurations.

In [ ]:
def interactive_monthly_evaluation(panels, battery):
    """
    Retrieves the pre-computed simulation results for the selected parameters
    and renders the monthly breakdown and summary table.
    """
    # Extract the target solution
    current_result = results_grid[(panels, battery)]

    # 1. Render the plot
    plot_monthly_breakdown(
        all_results={battery: current_result},
        battery_size=battery,
        pod_name=asset_name
    )

    # 2. Render the tabular summary (from the previous step)
    display_monthly_summary_table(current_result)

# Bind the UI controls to the parameter domains
interact(
    interactive_monthly_evaluation,
    panels=Dropdown(options=panel_domain, value=0, description='Panels:'),
    battery=Dropdown(options=battery_domain, value=0, description='Battery (kWh):')
)

With no panels all consumption comes from grid electricity.  
With some PV panels (but no battery) the generated energy covers some of the demand (green).  
Adding a battery allows much more energy demand to be covered (yellow).

Note that without a battery, even the maximum number of panels is not enough to cover grid consumption.  
It is more effective to have a small number of panels and a small battery than **just** the maximum number of panels.

## 4. Modelling the investment implications

The electricity bill has mainly three components. You could edit the values in the code cell below to match current tariffs.

**Grid import cost** = (variable rate × annual kWh from grid) + fixed annual fees  
Note this includes the energy fee (rate_energy) and the network transport fee (rate_network)  
**Feed-in revenue** = exported kWh × feed-in tariff — only earned when panels are installed  
**Feed-in subscription** = €5/month fixed cost to register as a seller — only paid when panels are installed

The variable rate and fixed fees are set by the network operator and reviewed annually. The feed-in tariff is set by regulation independently and has historically *decreased* as solar becomes more common.

In [ ]:
# --- Cost constants ---
rate_energy        =  0.13213
# Variable components (EUR/kWh)
rate_tax           =  0.001
rate_compensation  = -0.001
rate_network   =  0.051
var_rate_total     = rate_energy + rate_tax + rate_compensation + rate_network

# Fixed components (EUR/month)
fee_subscription   =  4.00
fee_remise         = -2.50
fee_metering       =  5.72
fee_network_fixed  =  7.42
fixed_monthly_total = fee_subscription + fee_remise + fee_metering + fee_network_fixed
fixed_annual_total  = fixed_monthly_total * 12

# Feed-in tariff (only applies when panels > 0)
rate_feedin        =  0.08815   # EUR/kWh exported to grid
fee_feedin_monthly =  5.00      # EUR/month, subscription to sell back
fee_feedin_annual  =  fee_feedin_monthly * 12


In [ ]:
baseline_kwh  = results_grid[(0, 0)]['grid'].sum()
variable_cost = baseline_kwh * var_rate_total
total_bill    = variable_cost + fixed_annual_total

def _row(label, detail, value, top_border=False,
         label_color="#111", detail_color="#555", value_color="#111"):
    border = "border-top: 2px solid #111;" if top_border else ""
    return (
        f"<tr style='{border}'>"
        f"<td style='padding:5px 18px;text-align:left;"
            f"color:{label_color};background-color:white'>{label}</td>"
        f"<td style='padding:5px 18px;text-align:center;"
            f"color:{detail_color};background-color:white'>{detail}</td>"
        f"<td style='padding:5px 18px;text-align:right;"
            f"color:{value_color};font-weight:bold;background-color:white'>{value}</td>"
        f"</tr>"
    )

html = f"""
<table style='border-collapse:collapse;font-size:13px;min-width:420px;background-color:white'>
  <caption style='font-weight:bold;font-size:14px;text-align:left;color:#111;
                  padding-bottom:8px;background-color:white'>
    Current Annual Bill &nbsp;—&nbsp;
    <span style='color:#555;font-weight:normal'>No PV · No Battery</span>
  </caption>
  <thead>
    <tr style='border-bottom:2px solid #111;background-color:white'>
      <th style='padding:5px 18px;text-align:left;color:#111'>Item</th>
      <th style='padding:5px 18px;text-align:center;color:#111'>Detail</th>
      <th style='padding:5px 18px;text-align:right;color:#111'>Value</th>
    </tr>
  </thead>
  <tbody>
    {_row("Grid consumption", "annual total",
          f"{baseline_kwh:,.0f} kWh",
          value_color="#111")}
    {_row("Variable rate",    "energy + network + taxes",
          f"{var_rate_total:.4f} €/kWh",
          detail_color="#555", value_color="#111")}
    {_row("Variable cost",    f"{baseline_kwh:,.0f} × {var_rate_total:.4f}",
          f"{variable_cost:,.2f} €",
          detail_color="#555", value_color="#d62728")}
    {_row("Fixed charges",    "metering + subscription + network",
          f"{fixed_annual_total:,.2f} €",
          detail_color="#555", value_color="#d62728")}
    {_row("Total Bill",       "",
          f"{total_bill:,.2f} €",
          top_border=True, value_color="#d62728")}
  </tbody>
</table>
"""

display(HTML(html))

In [ ]:
plot_parameter_space_heatmaps(
    results_grid,
    var_rate_total=var_rate_total,
    fixed_annual_total=fixed_annual_total,
    rate_feedin=rate_feedin,
    fee_feedin_annual=fee_feedin_annual,
)

In [ ]:
# --- Compute three metrics across all panel/battery configurations ---
# All values represent the CUMULATIVE 10-year position in euros.
# (0 panels, 0 battery) is NOT zero — it shows the total spend over 10 years.

import_cost_grid   = {}
feedin_revenue_grid = {}
net_position_grid  = {}

baseline_kwh = results_grid[(0, 0)]['grid'].sum()

for (panels, battery), sim_df in results_grid.items():

    # A. Total grid import cost over 10 years (price grows each year)
    grid_kwh = sim_df['grid'].sum()
    import_cost_10yr = sum(grid_kwh * p + fixed_annual_total
                           for p in price_forecast_2026_2035)

    # B. Feed-in revenue over 10 years (fixed tariff)
    if 'battery_charge' in sim_df.columns:
        batt_charge = sim_df['battery_charge'].sum()
    else:
        batt_charge = sim_df['battery_used'].sum()
    exported_kwh = max(0, sim_df['pv'].sum() - sim_df['pv_used'].sum() - batt_charge)
    feedin_revenue_10yr = exported_kwh * rate_feedin * 10
    feedin_sub_10yr     = fee_feedin_annual * 10 if panels > 0 else 0

    # C. Net position = total outgoings minus total income (negative = net cost)
    net_10yr = -(import_cost_10yr + feedin_sub_10yr - feedin_revenue_10yr)

    import_cost_grid[   (panels, battery)] = -import_cost_10yr      # negative = cost
    feedin_revenue_grid[(panels, battery)] =  feedin_revenue_10yr   # positive = income
    net_position_grid[  (panels, battery)] =  net_10yr

def _pivot(d):
    return pd.Series(d).unstack(level=0).sort_index(ascending=False).sort_index(axis=1)

pivot_import   = _pivot(import_cost_grid)   / 1000   # → €k
pivot_feedin   = _pivot(feedin_revenue_grid) / 1000  # → €k
pivot_net      = _pivot(net_position_grid)  / 1000   # → €k

In [ ]:
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
fig, axes = plt.subplots(2, 2, figsize=(20, 10))

def _diverging_cmap(pivot, ax, title, label):
    vmin = pivot.min().min()
    vmax = pivot.max().max()
    if vmin < 0 and vmax > 0:
        norm   = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        greens = plt.cm.Greens(np.linspace(0.2, 0.85, 128))
        reds   = plt.cm.Reds_r(np.linspace(0.2, 0.85, 128))
        cmap   = LinearSegmentedColormap.from_list("div", np.vstack([reds, greens]))
    elif vmax <= 0:
        norm, cmap = None, "Reds_r"
    else:
        norm, cmap = None, "Greens"
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".1f",
                cmap=cmap, norm=norm, annot_kws={"size": 9},
                cbar_kws={"label": label})
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Number of Panels")
    ax.set_ylabel("Battery Capacity (kWh)")

_diverging_cmap(
    pivot_import, axes[0, 0],
    "10-Year Grid Import Cost\n(incl. fixed fees)",
    "€k  (negative = cost)"
)
_diverging_cmap(
    pivot_feedin, axes[0, 1],
    "10-Year Feed-in Revenue\n(fixed tariff, excl. subscription)",
    "€k  (positive = income)"
)
_diverging_cmap(
    pivot_net, axes[1, 0],
    "10-Year Net Position\n(income minus all costs)",
    "€k  (negative = net cost)"
)

# Hide the unused fourth subplot
axes[1, 1].set_visible(False)

# Iterate correctly over all axes using .flat
for ax in axes.flat:
    if ax.get_visible():
        for _, spine in ax.spines.items():
            spine.set_visible(True)
            spine.set_linewidth(1.5)
            spine.set_edgecolor("black")

fig.suptitle(
    f"10-Year Financial Summary  |  PV scenario: {PV_SCENARIO_LABEL}  |  ",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

## 5. Refining the model by looking at energy price evolution

Luxembourg electricity prices were stable for nearly a decade, then the 2022 energy crisis drove market costs sharply higher. Government subsidies shielded consumers temporarily — absorbing roughly €0.06–0.08/kWh between 2022 and 2024. Those subsidies are now being withdrawn, and consumer prices are catching up to market reality.

**Why this matters for ROI:** a system that looks marginal at €0.20/kWh looks compelling at €0.29/kWh. The forecast you set below directly controls the 10-year savings calculation.

### Your Price Forecast

The chart below anchors to the actual Luxembourg electricity price history. Your task is to forecast what the variable rate will do over the next 10 years.

<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Use the two sliders to forecast electricity prices (then run the cells below)
</div>


- **2026 Start Price** — where you expect the price to be next year. The default is the current 2025 rate (€0.29/kWh). You might adjust this upward if you believe the subsidy withdrawal is not yet complete.
- **Annual Growth Rate** — the year-on-year percentage change applied geometrically. At 3% the price reaches ~€0.39/kWh by 2035. At 0% it stays flat. Negative values model a price decrease.

There is no objectively correct answer. The point is to understand how sensitive the 10-year savings calculation is to your assumptions — run the savings heatmap afterwards with an optimistic forecast, then a pessimistic one, and compare.

In [ ]:
from IPython.display import display
# HTML, FloatSlider, Output, VBox already imported from ipywidgets at top of notebook

current_year        = 2025
forecast_start_year = current_year + 1
forecast_end_year   = forecast_start_year + 9  # 10-year window

growth_rate_slider = FloatSlider(
    value=2.0, min=-5.0, max=15.0, step=0.5,
    description='Grid price growth (%/yr):',
    style={'description_width': 'initial'}, layout={'width': '450px'}
)
feedin_rate_slider = FloatSlider(
    value=-1.0, min=-10.0, max=5.0, step=0.5,
    description='Feed-in rate change (%/yr):',
    style={'description_width': 'initial'}, layout={'width': '450px'}
)

output_plot = Output()
price_forecast_2026_2035  = []
feedin_forecast_2026_2035 = []

def update_forecast(change=None):
    global price_forecast_2026_2035, feedin_forecast_2026_2035

    grid_rate   = growth_rate_slider.value / 100
    feedin_rate = feedin_rate_slider.value / 100
    future_years = np.arange(forecast_start_year, forecast_end_year + 1)

    # Anchor on current rates, grow from year 1
    grid_prices   = [var_rate_total * (1 + grid_rate)**i   for i in range(len(future_years))]
    feedin_prices = [rate_feedin    * (1 + feedin_rate)**i for i in range(len(future_years))]

    price_forecast_2026_2035  = grid_prices
    feedin_forecast_2026_2035 = feedin_prices

    with output_plot:
        output_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        # Anchor points
        ax.scatter([current_year], [var_rate_total], color='#d62728', zorder=5)
        ax.scatter([current_year], [rate_feedin],    color='#2ca02c', zorder=5)

        # Forecast lines
        ax.plot(future_years, grid_prices,
                marker='o', color='#d62728', linewidth=2.5,
                label=f'Grid purchase price  ({growth_rate_slider.value:+.1f}%/yr)')
        ax.plot(future_years, feedin_prices,
                marker='o', color='#2ca02c', linewidth=2.5,
                label=f'Feed-in tariff  ({feedin_rate_slider.value:+.1f}%/yr)')

        # Annotate end values
        ax.annotate(f"{grid_prices[-1]:.3f} €",
                    xy=(future_years[-1], grid_prices[-1]),
                    xytext=(6, 0), textcoords='offset points',
                    va='center', color='#d62728', fontsize=10)
        ax.annotate(f"{feedin_prices[-1]:.3f} €",
                    xy=(future_years[-1], feedin_prices[-1]),
                    xytext=(6, 0), textcoords='offset points',
                    va='center', color='#2ca02c', fontsize=10)

        # Current-year anchor labels
        ax.annotate(f"{var_rate_total:.4f} €  (now)",
                    xy=(current_year, var_rate_total),
                    xytext=(-6, 6), textcoords='offset points',
                    ha='right', color='#d62728', fontsize=9)
        ax.annotate(f"{rate_feedin:.4f} €  (now)",
                    xy=(current_year, rate_feedin),
                    xytext=(-6, -10), textcoords='offset points',
                    ha='right', color='#2ca02c', fontsize=9)

        ax.set_title("10-Year Price Forecast", fontsize=13)
        ax.set_ylabel("€ per kWh")
        ax.set_xlabel("Year")
        ax.set_xlim(current_year - 0.5, forecast_end_year + 0.8)
        ax.set_ylim(0, max(max(grid_prices), max(feedin_prices)) * 1.5)
        ax.xaxis.set_major_locator(plt.MultipleLocator(1))
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.legend(loc='upper left')
        ax.text(current_year - 0.4, 0.005,
                "*Excluding fixed annual fees", fontsize=9, color='gray')

        plt.tight_layout()
        plt.show()

growth_rate_slider.observe(update_forecast, names='value')
feedin_rate_slider.observe(update_forecast, names='value')

display(VBox([
    HTML(f"<h3>Step 1: Forecast Variable Rates (€/kWh)</h3>"
         f"<p style='margin:0;font-size:12px;color:#555'>"
         f"Grid purchase anchored at <b style='color:#d62728'>{var_rate_total:.4f} €/kWh</b> &nbsp;·&nbsp; "
         f"Feed-in tariff anchored at <b style='color:#2ca02c'>{rate_feedin:.4f} €/kWh</b><br>"
         f"<span style='font-size:11px'>Fixed annual charges (~€{fixed_annual_total:.0f}/yr) excluded here, added separately.</span>"
         f"</p>"),
    growth_rate_slider, feedin_rate_slider, output_plot
]))

update_forecast()

We assume demand will be the same as the `TARGET_YEAR` energy consumption data being used throughout.  
**Your** future electricity price forecast would result in annual bills as follows:

In [ ]:
# 1. Base case annual bill (0 panels, 0 battery)
base_annual_bill = calculate_annual_bill(
    results_grid[(0, 0)],
    num_panels=0,
    var_rate_total=var_rate_total,
    fixed_annual_total=fixed_annual_total,
    rate_feedin=rate_feedin,
    fee_feedin_annual=fee_feedin_annual,
)


In [ ]:
# --- Baseline annual bill for each forecast year ---
baseline_kwh = results_grid[(0, 0)]['grid'].sum()
baseline_bills = [baseline_kwh * p + fixed_annual_total
                  for p in price_forecast_2026_2035]
forecast_years = list(range(2026, 2036))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(forecast_years, baseline_bills, color="tomato", alpha=0.8, edgecolor="black", linewidth=0.5)
ax.plot(forecast_years, baseline_bills, color="darkred", marker="o", linewidth=2)

for yr, bill in zip(forecast_years, baseline_bills):
    ax.text(yr, bill + 10, f"€{bill:.0f}", ha="center", va="bottom", fontsize=9)

ax.set_title("Projected Annual Electricity Bill — No PV, No Battery", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Annual Bill (€)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Assessing the investment: 10-Year Financial Results

The heatmaps below use the price forecast you just set. All savings figures are calculated using your assumed trajectory — not a fixed assumption.

**If you want to try a different forecast:** go back to the forecasting slider cell, adjust the values, then re-run all cells below that one.

In [ ]:
# --- Compute three metrics across all panel/battery configurations ---
# All values represent the CUMULATIVE 10-year position in euros.
# (0 panels, 0 battery) is NOT zero — it shows the total spend over 10 years.

import_cost_grid   = {}
feedin_revenue_grid = {}
net_position_grid  = {}

baseline_kwh = results_grid[(0, 0)]['grid'].sum()

for (panels, battery), sim_df in results_grid.items():

    # A. Total grid import cost over 10 years (price grows each year)
    grid_kwh = sim_df['grid'].sum()
    import_cost_10yr = sum(grid_kwh * p + fixed_annual_total
                           for p in price_forecast_2026_2035)

    # B. Feed-in revenue over 10 years (fixed tariff)
    if 'battery_charge' in sim_df.columns:
        batt_charge = sim_df['battery_charge'].sum()
    else:
        batt_charge = sim_df['battery_used'].sum()
    exported_kwh = max(0, sim_df['pv'].sum() - sim_df['pv_used'].sum() - batt_charge)
    feedin_revenue_10yr = sum(exported_kwh * fp for fp in feedin_forecast_2026_2035)
    feedin_sub_10yr     = fee_feedin_annual * 10 if panels > 0 else 0

    # C. Net position = total outgoings minus total income (negative = net cost)
    net_10yr = -(import_cost_10yr + feedin_sub_10yr - feedin_revenue_10yr)

    import_cost_grid[   (panels, battery)] = -import_cost_10yr      # negative = cost
    feedin_revenue_grid[(panels, battery)] =  feedin_revenue_10yr   # positive = income
    net_position_grid[  (panels, battery)] =  net_10yr

def _pivot(d):
    return pd.Series(d).unstack(level=0).sort_index(ascending=False).sort_index(axis=1)

pivot_import   = _pivot(import_cost_grid)   / 1000   # → €k
pivot_feedin   = _pivot(feedin_revenue_grid) / 1000  # → €k
pivot_net      = _pivot(net_position_grid)  / 1000   # → €k

In [ ]:
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
fig, axes = plt.subplots(2, 2, figsize=(20, 10))

def _diverging_cmap(pivot, ax, title, label):
    vmin = pivot.min().min()
    vmax = pivot.max().max()
    if vmin < 0 and vmax > 0:
        norm   = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        greens = plt.cm.Greens(np.linspace(0.2, 0.85, 128))
        reds   = plt.cm.Reds_r(np.linspace(0.2, 0.85, 128))
        cmap   = LinearSegmentedColormap.from_list("div", np.vstack([reds, greens]))
    elif vmax <= 0:
        norm, cmap = None, "Reds_r"
    else:
        norm, cmap = None, "Greens"
    sns.heatmap(pivot, ax=ax, annot=True, fmt=".1f",
                cmap=cmap, norm=norm, annot_kws={"size": 9},
                cbar_kws={"label": label})
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Number of Panels")
    ax.set_ylabel("Battery Capacity (kWh)")

_diverging_cmap(
    pivot_import, axes[0, 0],
    "10-Year Grid Import Cost\n(incl. fixed fees)",
    "€k  (negative = cost)"
)
_diverging_cmap(
    pivot_feedin, axes[0, 1],
    "10-Year Feed-in Revenue\n(fixed tariff, excl. subscription)",
    "€k  (positive = income)"
)
_diverging_cmap(
    pivot_net, axes[1, 0],
    "10-Year Net Position\n(income minus all costs)",
    "€k  (negative = net cost)"
)

# Hide the unused fourth subplot
axes[1, 1].set_visible(False)

# Iterate correctly over all axes using .flat
for ax in axes.flat:
    if ax.get_visible():
        for _, spine in ax.spines.items():
            spine.set_visible(True)
            spine.set_linewidth(1.5)
            spine.set_edgecolor("black")

fig.suptitle(
    f"10-Year Financial Summary  |  PV scenario: {PV_SCENARIO_LABEL}  |  "
    f"Price forecast: {price_forecast_2026_2035[0]:.2f} → {price_forecast_2026_2035[-1]:.2f} €/kWh",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

### Installation Costs & Subsidies

The Luxembourg state offers capital subsidies for residential PV and battery installations.


<div style="padding: 10px; border-left: 5px solid #41e1a4; background-color: rgba(65, 225, 164, 0.1); border-radius: 4px;">
<b style="color: #2eb886;">✎ Action Required:</b> Set the sliders to reflect current quotes and the applicable subsidy rate. The heatmap updates immediately.
</div>

Green cells are configurations where 10-year savings exceed total net cost — financially viable. Red cells have not yet paid back. The boundary between them is your break-even line.

In [ ]:
fixed_cost_slider   = IntSlider(value=8000, min=0,   max=10000, step=500,
                                description='Fixed Setup (€):',
                                style={'description_width': 'initial'}, layout={'width': '450px'})
panel_cost_slider   = IntSlider(value=300,  min=100, max=1000,  step=50,
                                description='Cost per Panel (€):',
                                style={'description_width': 'initial'}, layout={'width': '450px'})
battery_cost_slider = IntSlider(value=500,  min=200, max=1500,  step=50,
                                description='Cost per kWh Bat (€):',
                                style={'description_width': 'initial'}, layout={'width': '450px'})
subsidy_slider      = FloatSlider(value=50.0, min=0.0, max=75, step=5,
                                  description='Subsidy (%):',
                                  style={'description_width': 'initial'}, layout={'width': '450px'})
output_abs = Output()

def update_absolute_heatmap(change=None):
    subsidy_multiplier = 1 - (subsidy_slider.value / 100.0)

    # --- Build CAPEX matrix ---
    capex_grid = {}
    for (panels, battery) in results_grid.keys():
        if panels == 0 and battery == 0:
            capex_grid[(panels, battery)] = 0
        else:
            gross = (fixed_cost_slider.value
                     + panels * panel_cost_slider.value
                     + battery * battery_cost_slider.value)
            capex_grid[(panels, battery)] = gross * subsidy_multiplier

    capex_df = (pd.Series(capex_grid)
                .unstack(level=0)
                .sort_index(ascending=False)
                .sort_index(axis=1)) / 1000  # → €k

    with output_abs:
        output_abs.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(22, 6))

        # --- Left: installation CAPEX ---
        sns.heatmap(
            capex_df, ax=axes[0],
            annot=True, fmt=".1f", cmap="Oranges",
            annot_kws={"size": 13},
            cbar_kws={"label": "Net Installation Cost (€k)"}
        )
        axes[0].set_title(
            f"Installation Cost After {subsidy_slider.value:.0f}% Subsidy (€k)",
            fontsize=13, pad=12
        )
        axes[0].set_xlabel("Number of Panels")
        axes[0].set_ylabel("Battery Capacity (kWh)")

        # --- Right: 10-year absolute financial position ---
        plot_absolute_position_heatmap(
            results_grid         = results_grid,
            price_forecast       = price_forecast_2026_2035,
            feedin_forecast      = feedin_forecast_2026_2035,
            fixed_cost           = fixed_cost_slider.value,
            panel_cost           = panel_cost_slider.value,
            battery_cost_per_kwh = battery_cost_slider.value,
            subsidy_pct          = subsidy_slider.value,
            fixed_annual_total   = fixed_annual_total,
            fee_feedin_annual    = fee_feedin_annual,
            ax                   = axes[1],   # inject into right subplot
        )

        plt.tight_layout()
        plt.show()
        plt.close(fig)

for w in [fixed_cost_slider, panel_cost_slider, battery_cost_slider, subsidy_slider]:
    w.observe(update_absolute_heatmap, names='value')

display(VBox([
    HTML("<h3>Step 2: Financial Viability Analysis</h3>"),
    HTML("Set the installation costs and the level of state subsidy:"),
    fixed_cost_slider, panel_cost_slider, battery_cost_slider, subsidy_slider,
    output_abs
]))

update_absolute_heatmap()

In [ ]:
# --- Final cell (investment metrics) ---
output_metrics = Output()

def update_metrics_heatmaps(change=None):
    with output_metrics:
        output_metrics.clear_output(wait=True)
        plot_investment_metrics_heatmaps(
            results_grid         = results_grid,
            price_forecast       = price_forecast_2026_2035,
            feedin_forecast      = feedin_forecast_2026_2035,   # ← replaces rate_feedin
            fixed_cost           = fixed_cost_slider.value,
            panel_cost           = panel_cost_slider.value,
            battery_cost_per_kwh = battery_cost_slider.value,
            subsidy_pct          = subsidy_slider.value,
            fixed_annual_total   = fixed_annual_total,
            fee_feedin_annual    = fee_feedin_annual,
            discount_rate        = 0.03,
            pv_scenario_label    = PV_SCENARIO_LABEL,
        )

for w in [fixed_cost_slider, panel_cost_slider, battery_cost_slider, subsidy_slider]:
    w.observe(update_metrics_heatmaps, names='value')

display(VBox([output_metrics]))
update_metrics_heatmaps()

---

## What We Have Learned

Working through this household's data, some conclusions stand out.

**1. Consumption timing matters as much as consumption volume.**  
The family uses most electricity in the morning and evening — but solar generation peaks at
midday. A PV system alone will cover a useful fraction of their annual demand, but a
significant portion of its production will either be exported to the grid at a low feed-in
rate or simply go unused. The hourly profile plots make this mismatch visible.

**2. A battery does not generate energy — it shifts it.**  
Adding battery storage does not change how much solar is produced. It changes *when*
that energy is available, moving midday surplus into the evening peak. The weekly energy
flow charts showed this clearly. With 20 panels, in Spring the battery fills during the day and
empties by 10pm; in winter it barely fills at all, because there is little surplus to store.

**3. Solar panels do not eliminate the winter grid bill.**  
Even the most generous panel configurations leave this household heavily grid-dependent
in the November–February period. The monthly breakdown charts show that winter grid
consumption is largely irreducible — no realistic roof-mounted PV system in Luxembourg
can offset it. This is not a failure of the technology; it is the physics of solar
irradiation at 49°N latitude.

**4. The financial case is sensitive to subsidies and price assumptions.**  
At the current 2026 tariff rate of ~€0.29/kWh, modest PV configurations (20-30 panels) plus 10kWh battery
typically break even within 10 years. If prices
continue rising at 3–5% per year, the case strengthens considerably. If prices fall or
stagnate, the payback period extends. The single most important variable is the one we
cannot know...future electricity prices.

---

### Taking This Further

This notebook modelled a single household with a single roof. The same methodology applies
directly to schools, community buildings, and small commercial premises — with the main
difference being the tariff structure. These buildings often face a peak demand charge
which batteries can directly reduce.

Key questions for any real installation decision:
- What does an installer quote for this roof, today?
- What is the current Luxembourg subsidy rate?
- Does the household expect to add an electric vehicle? A heat-pump? If so, the self-consumption calculation changes substantially.
- Is the priority financial return, energy independence, or carbon reduction?
The optimal configuration differs depending on the answer.

For this household, in this location, at current prices, a PV installation makes financial sense. Their daily activity schedule means that a battery is crucial to enable self-consumption of the energy produced (instead of it all going to feed-in at a low rate).